# 2. Вопросы к изображению

Классификатор из первого задания выбирает один из фиксированных видов. Здесь дадим
модели картинку **и вопрос**: например, «Белая ли грудка?» или «Есть ли полоски на крыльях?».
Ответ зависит от обоих входов. Будем дообучать Qwen3.5-0.8B отвечать `yes` или `no`.

**VLM** (*vision-language model*) — модель, работающая с изображением и языком.
Визуальный энкодер превращает участки картинки в векторы признаков; затем эти признаки
передаются языковой части вместе с токенами вопроса. Языковая модель генерирует ответ
последовательно, предсказывая следующий токен по изображению и уже известному тексту.

![Как изображение и вопрос превращаются в ответ](../assets/vlm.png)

Сначала реализуем маленький LoRA-слой, затем разберём подготовку входов и допишем
маску функции потерь. После этого сравним ответы модели до и после обучения.

In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
ROOT = Path.cwd()
if ROOT.name in {"notebooks", "solutions"}:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("BIRD_DATA", str(ROOT / "data/cub8")))
assert (DATA / "manifest.jsonl").exists(), "Run scripts/prepare_data.py first"
import torch
from torch import nn
torch.manual_seed(42)
torch.set_num_threads(4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

## TODO 1 — LoRALinear

Для небольшой задачи не обязательно изменять все веса большой модели.
**LoRA** (*Low-Rank Adaptation*) оставляет исходную матрицу $W_0$ замороженной
и обучает добавку низкого ранга: $\Delta W=(\alpha/r)BA$.
Сначала $A$ переводит вход в пространство размерности $r$, затем $B$ возвращает
его в выходное пространство. При небольшом $r$ обучаемых параметров гораздо меньше,
чем в полной матрице; $\alpha/r$ задаёт масштаб добавки.

$y=W_0x+b+(\alpha/r)BAx$, где $A\in R^{r\times d_{in}}$, $B\in R^{d_{out}\times r}$.

Заморозьте base, инициализируйте A случайно, B нулями. Напишите forward и merged().
Метод `merged()` должен вернуть обычный линейный слой с весами $W_0+\Delta W$:
при применении модели отдельная ветка LoRA тогда не нужна.

До запуска предскажите: какой градиент на первом шаге равен нулю? Почему обе матрицы
нельзя занулить? Сколько параметров при d_in=d_out=1024 и r=8?

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank=4, alpha=4):
        super().__init__()
        raise NotImplementedError("TODO: frozen base, A, B, scaling")
    def forward(self, x):
        raise NotImplementedError("TODO: low-rank update")
    def merged(self):
        raise NotImplementedError("TODO: return equivalent nn.Linear")

In [ ]:
base = nn.Linear(5, 3)
adapter = LoRALinear(base, rank=2)
x = torch.randn(4, 5)
torch.testing.assert_close(adapter(x), base(x))
adapter(x).square().sum().backward()
assert adapter.A.grad.abs().sum() == 0
assert adapter.B.grad.abs().sum() > 0
assert base.weight.grad is None
with torch.no_grad():
    adapter.B.add_(.1)
torch.testing.assert_close(adapter(x), adapter.merged()(x))

## От одного слоя к VLM

В проверке выше мы явно построили одну низкоранговую добавку. В большой модели таких
слоёв много: библиотека PEFT установит аналогичные адаптеры в линейные слои языковой
части. Визуальный энкодер и выходная матрица словаря останутся замороженными.

Из каждой фотографии получаем несколько пар «вопрос — ответ» по её разметке.
В validation используются другие формулировки тех же вопросов. Посмотрим на один
пример и на число параметров, которые действительно будут обучаться.

In [ ]:
from birdlab.data import make_qa
from birdlab.vlm import load_model, QACollator, evaluate, messages
train_qa = make_qa(DATA, "train", balance=True)
val_qa = make_qa(DATA, "val", heldout_wording=True)
print(len(train_qa), len(val_qa), train_qa[0])
from transformers import set_seed
set_seed(42)
model, processor = load_model(target_mode=os.environ.get("BIRD_TARGETS", "all-linear"))

## Processor и токенизация

Текстовая модель получает не строки, а **токены** — элементы своего словаря,
обозначенные целочисленными IDs. Токен может соответствовать слову, его части или
служебному маркеру. **Токенайзер** преобразует текст в IDs и обратно.

У VLM есть также **processor**: он объединяет токенайзер с подготовкой изображений.
А **chat template** оформляет диалог: отмечает роли пользователя и ассистента,
место изображения и начало ответа. Поэтому недостаточно просто склеить вопрос с `yes`.

Рассмотрите IDs, токены и обратное декодирование. Совпадает ли число токенов с числом слов?
Посмотрите `pixel_values` и `image_grid_thw`. Почему картинку не обрабатывает tokenizer?

In [ ]:
for text in ["yes", "no", "house sparrow", "домовый воробей"]:
    ids = processor.tokenizer.encode(text, add_special_tokens=False)
    print(text, ids, processor.tokenizer.convert_ids_to_tokens(ids))
example = train_qa[0]
print(processor.apply_chat_template(messages(example, True), tokenize=False, enable_thinking=False))

## TODO 2 — маска loss

Во время обучения подаём весь диалог, включая правильный ответ. На каждой позиции
модель предсказывает следующий токен, видя предыдущие правильные токены — это
*teacher forcing*. Сдвиг входов и целей уже выполняет модель.

Мы хотим учить её отвечать, поэтому считаем loss только на ответе, включая маркер
его окончания (**EOS**). Вопрос и заполнители (**padding**), выравнивающие длины
примеров в батче, в loss не входят. Значение `-100` в `labels` означает «игнорировать
эту позицию»; сам вход при этом остаётся доступен модели.

$L=-\sum_t m_t\log p(y_t|I,q,y_{<t})/\sum_t m_t$.
Здесь $I$ — изображение, $q$ — вопрос, а $m_t=1$ только для токенов ответа.

Верните копию input_ids; prompt и padding замените на -100. EOS ответа сохраняется.
Используйте attention_mask, а не равенство pad_token_id: pad и EOS могут совпадать.

In [ ]:
def answer_labels(input_ids, attention_mask, prompt_lengths):
    raise NotImplementedError("TODO: mask prompt and padding, preserve answer/EOS")

In [ ]:
ids = torch.tensor([[1, 2, 7, 9, 9], [1, 2, 3, 8, 9]])
attention = torch.tensor([[1, 1, 1, 1, 0], [1, 1, 1, 1, 1]])
assert answer_labels(ids, attention, [2, 3]).tolist() == [[-100,-100,7,9,-100],[-100,-100,-100,8,9]]
collator = QACollator(processor, label_function=answer_labels)
batch = collator(train_qa[:2])
for key, value in batch.items():
    print(key, tuple(value.shape))
for labels in batch["labels"]:
    print("Loss on:", processor.decode(labels[labels != -100]))
loss = model(**{k:v.to(DEVICE) for k,v in batch.items()}).loss
loss.backward()
assert any(p.grad is not None and p.grad.abs().sum() > 0 for n,p in model.named_parameters() if "lora_B" in n)
model.zero_grad(set_to_none=True)
del batch, loss

## До и после LoRA

Сравните ответы до обучения и после 50 шагов LoRA.
Один шаг оптимизатора здесь объединяет два батча по четыре примера. Проверяем именно
сгенерированные ответы: хороший loss на правильном продолжении ещё не гарантирует,
что при самостоятельной генерации модель выберет нужный ответ.

Для быстрой проверки берём 24 вопроса. Для содержательного сравнения задайте
`BIRD_EVAL_SIZE=160`: это включает все доступные validation-вопросы.

In [ ]:
import random
random.Random(42).shuffle(val_qa)
evaluation = val_qa[:int(os.environ.get("BIRD_EVAL_SIZE", "24"))]
before = evaluate(model, processor, evaluation)
print(before["tasks"])
from transformers import Trainer, TrainingArguments
RUN = ROOT / "runs/notebook_vlm"
args = TrainingArguments(output_dir=str(RUN), max_steps=int(os.environ.get("BIRD_STEPS", "50")), learning_rate=2e-4,
    per_device_train_batch_size=4, gradient_accumulation_steps=2,
    remove_unused_columns=False, report_to="none", save_strategy="no", logging_steps=5,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    gradient_checkpointing=False)
model.config.use_cache = False
trainer = Trainer(model=model, args=args, train_dataset=train_qa, data_collator=collator)
trainer.train()
model.config.use_cache = True
after = evaluate(model, processor, evaluation)
print(after["tasks"])
model.save_pretrained(RUN / "adapter")
processor.save_pretrained(RUN / "adapter")

## Эксперименты

Не на все вопросы ответы `yes` и `no` встречаются одинаково часто. Поэтому наряду
с accuracy смотрим balanced accuracy по каждому вопросу и среднее этих значений.
Это помогает заметить модель, которая просто выбирает более частый ответ.

1. Для каждого вопроса выберите самый частый ответ в train и оцените такую константную
   модель на validation. Насколько LoRA улучшает этот результат?
2. Перемешайте изображения, сохранив вопросы и правильные ответы. Если качество
   почти не изменилось, что это говорит об использовании картинки?
3. Измените ранг в `load_model` и сравните r=2 и r=8 при одинаковом числе шагов.
   Как меняются число параметров, время и качество?

Сохраняем только адаптер: для его загрузки понадобится та же исходная модель.
Проверим, что новый экземпляр с сохранённым адаптером воспроизводит ответ.

In [ ]:
from birdlab.vlm import predict, MODEL_ID
from transformers import Qwen3_5ForConditionalGeneration
from peft import PeftModel
# Save a reference before releasing GPU memory; reload exactly the saved adapter.
reference = predict(model, processor, evaluation[0])
dtype = next(model.parameters()).dtype
del trainer, model
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
base = Qwen3_5ForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype, attn_implementation="sdpa").to(DEVICE)
restored = PeftModel.from_pretrained(base, RUN / "adapter")
assert predict(restored, processor, evaluation[0]) == reference
print("Adapter reload OK:", reference)